# Lab notebook — PCA on the Brazilian PRE curve (issue #7)

Exploration, not the post. Goal: reach the **target figure** (loadings of PC1–PC3 + explained variance)
and check whether the result is robust. Messy is fine; the post (`index.qmd`) is written later from what survives here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find the repo root (folder with _quarto.yml), so paths work wherever the notebook runs
ROOT = Path.cwd()
while not (ROOT / "_quarto.yml").exists():
    ROOT = ROOT.parent

# Categorical slots 1-3 (validated default palette); markers/line styles keep PCs distinguishable without color
COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
STYLES = [dict(marker="o", ls="-"), dict(marker="s", ls="--"), dict(marker="^", ls=":")]
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})

START = "2006-01-01"      # all tenors 3M-10Y available from here (10Y starts 2005-09-29)

## 1. Data

In [ ]:
curve = pd.read_parquet(ROOT / "data/processed/br_pre_curve_cm.parquet").loc[START:]
X_POS = np.arange(curve.shape[1])   # tenors plotted at equal spacing so short-end labels do not collide

changes = (curve.diff() * 100).dropna()        # daily changes in basis points
print(f"{curve.index[0].date()} -> {curve.index[-1].date()}: {len(curve):,} days, {len(changes):,} daily changes")
print("Missing values:", int(curve.isna().sum().sum()))
curve.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for col, color, style in zip(["3M", "2Y", "10Y"], COLORS, STYLES):
    ax.plot(curve.index, curve[col], color=color, lw=1.2, ls=style["ls"], label=col)
ax.set_ylabel("% p.a. (base 252)")
ax.set_title("PRE curve, constant maturities")
ax.legend(loc="upper right", frameon=False, ncols=3)
plt.show()

## 2. Data check: extreme days

PCA on a covariance matrix is driven by the largest moves, so look at them first.
Most should match known events (2008 crisis, Joesley Day 2017-05-18, Covid March 2020, 2022-11-10).

⚠️ **To verify:** the October 2026 moves (2026-10-05 and 2026-10-26) are the largest in the sample.
They fall right after the two rounds of the 2026 election, so a real shock is plausible —
but cross-check them against the DI1 contract settlements you have locally (e.g. DI1F28, DI1F31) before trusting them.

In [ ]:
extreme = changes[(changes.abs() > 75).any(axis=1)]
print(f"{len(extreme)} days with some tenor moving more than 75 bp")
extreme.round(0)

## 3. PCA

In [ ]:
def pca(X: pd.DataFrame, standardize: bool = False):
    """Eigen-decomposition of the covariance (or correlation) matrix of X."""
    Z = (X - X.mean()) / X.std() if standardize else X - X.mean()
    eigvals, eigvecs = np.linalg.eigh(np.cov(Z.values.T))
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Sign convention so the PCs read naturally:
    # PC1 > 0 on average (rates up), PC2 > 0 at the long end (steepening), PC3 > 0 at the belly (2Y-3Y)
    for k, anchor in [(0, slice(None)), (1, -1), (2, slice(3, 5))]:
        if np.mean(eigvecs[anchor, k]) < 0:
            eigvecs[:, k] *= -1

    loadings = pd.DataFrame(eigvecs, index=X.columns, columns=[f"PC{i+1}" for i in range(X.shape[1])])
    explained = pd.Series(eigvals / eigvals.sum(), index=loadings.columns)
    scores = Z @ eigvecs
    scores.columns = loadings.columns
    return loadings, explained, scores

load_cov, expl_cov, scores = pca(changes)
load_corr, expl_corr, _ = pca(changes, standardize=True)

pd.DataFrame({"covariance": expl_cov, "correlation": expl_corr}).head(4).style.format("{:.1%}")

### Target figure

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})

for k, (color, style) in enumerate(zip(COLORS, STYLES)):
    pc = f"PC{k+1}"
    ax1.plot(X_POS, load_cov[pc], color=color, lw=2, ms=6, **style,
             label=f"{pc} ({expl_cov[pc]:.1%})")
ax1.axhline(0, color="grey", lw=0.8)
ax1.set_xticks(X_POS, curve.columns)
ax1.set_xlabel("Tenor")
ax1.set_ylabel("Loading")
ax1.set_title("Loadings (covariance of daily changes, bp)")
ax1.legend(frameon=False)

top = expl_cov.head(5)
ax2.bar(top.index, top.values * 100, color=COLORS[0], width=0.6)
for x, v in zip(top.index, top.values):
    ax2.annotate(f"{v:.1%}", (x, v * 100), xytext=(0, 3), textcoords="offset points",
                 ha="center", fontsize=9)
ax2.set_ylabel("% of variance")
ax2.set_title("Explained variance")
plt.tight_layout()
plt.show()

**First read (fill in):**
- PC1 = level? Is it flat across tenors, or small at the short end (front anchored by Selic expectations)?
- PC2 = slope? Where does it cross zero?
- PC3 = curvature? Where is the hump?
- Compare shapes with Carol Alexander, Vol. II, PCA chapter.

In [ ]:
# Covariance vs. correlation loadings: does standardizing change the story?
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for k, ax in enumerate(axes):
    pc = f"PC{k+1}"
    ax.plot(X_POS, load_cov[pc], color=COLORS[0], lw=2, marker="o", label="covariance")
    ax.plot(X_POS, load_corr[pc], color=COLORS[1], lw=2, marker="s", ls="--", label="correlation")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xticks(X_POS, curve.columns, fontsize=8)
    ax.set_title(pc)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 4. Sanity check: do the PCs really mean level, slope and curvature?

In [ ]:
# Cumulated PC scores vs. simple curve measures (changes over the same days)
proxies = pd.DataFrame({
    "level (mean of tenors)": changes.mean(axis=1),
    "slope (10Y - 3M)": changes["10Y"] - changes["3M"],
    "curvature (2*2Y - 3M - 10Y)": 2 * changes["2Y"] - changes["3M"] - changes["10Y"],
})
pd.concat([scores[["PC1", "PC2", "PC3"]], proxies], axis=1).corr().loc[proxies.columns, ["PC1", "PC2", "PC3"]].round(2)

## 5. Stability over time

In [ ]:
WINDOW = 252   # 1 year of business days

def share_top(X: pd.DataFrame) -> pd.Series:
    w = np.sort(np.linalg.eigvalsh(np.cov((X - X.mean()).values.T)))[::-1]
    w = w / w.sum()
    return pd.Series({"PC1": w[0], "PC1-2": w[:2].sum(), "PC1-3": w[:3].sum()})

ends = changes.index[WINDOW::21]   # roughly monthly, to keep it fast
rolling = pd.DataFrame({d: share_top(changes.loc[:d].iloc[-WINDOW:]) for d in ends}).T

fig, ax = plt.subplots(figsize=(9, 3.8))
for col, color, style in zip(rolling.columns, COLORS, STYLES):
    ax.plot(rolling.index, rolling[col] * 100, color=color, lw=1.5, ls=style["ls"], label=col)
ax.set_ylabel("% of variance (1-year window)")
ax.set_title("How much of the curve's movement the first PCs explain, over time")
ax.legend(frameon=False, loc="lower left")
plt.show()

In [ ]:
# Rolling PC1 loadings: does the 'level' factor keep its shape?
pc1_roll = pd.DataFrame({d: pca(changes.loc[:d].iloc[-WINDOW:])[0]["PC1"] for d in list(ends[::12]) + [changes.index[-1]]}).T

fig, ax = plt.subplots(figsize=(9, 3.8))
for d, row in pc1_roll.iterrows():
    ax.plot(X_POS, row.values, color=COLORS[0], alpha=0.25 + 0.75 * (d == pc1_roll.index[-1]),
            lw=2 if d == pc1_roll.index[-1] else 1)
ax.plot([], [], color=COLORS[0], lw=1, alpha=0.3, label="1-year windows, yearly")
ax.plot([], [], color=COLORS[0], lw=2, label=f"last window ({pc1_roll.index[-1].date()})")
ax.set_xticks(X_POS, curve.columns)
ax.set_ylabel("PC1 loading")
ax.set_title("PC1 loadings across time")
ax.legend(frameon=False)
plt.show()

## 6. Robustness: without the extreme days

In [ ]:
cut = changes.abs().max(axis=1).quantile(0.99)
calm = changes[changes.abs().max(axis=1) <= cut]
_, expl_calm, _ = pca(calm)
pd.DataFrame({"all days": expl_cov, f"without top 1% days (> {cut:.0f} bp)": expl_calm}).head(3).style.format("{:.1%}")

## Notes for the post

- Result:
- Surprises:
- Caveats:
- Next questions (→ new issues):